# PKG Geographic Analytics — EDA & Use-Case Notebook
### Payment Knowledge Graph · Treasury Management · Data Science
**Scope: `version = 'P99_9'` · table `bdahd01p_dlcdi1_cdi_tm.cust_c2c_metrics`**

---

## What this notebook is for

Block F emits 24 geographic columns per node per month. This notebook answers
three questions in order, and refuses to skip to the third:

1. **Is the geo block trustworthy?** Coverage gates, guard-NaN rates, and the
   population composition recomputed under `party_type`.
2. **Does geography carry information that the other metrics don't already
   have?** Most candidate "geographic findings" are size restated. The peer
   normalisation layer (§4) exists to strip that out; the correlation
   structure (§5) shows what survives.
3. **What is worth building an application around?** Four candidate use cases
   (§7–§10), each with a persistence requirement and a rung-agreement check.

## Standing constraints this notebook enforces

| Constraint | Where enforced |
|---|---|
| `scope = on_us_c2c` — net flow means *vs other PNC customers*, not vs the economy | stated on every aggregate |
| Never aggregate across `node_type` without declaring it | every groupby carries it |
| Nothing reportable on a single month | §6 persistence, §7–9 require *k* consecutive months |
| Nothing reportable at one rung — check two | §11 P99_9 vs P99 |
| Geo metrics are computed on **located counterparties only** | §2 coverage gate, applied everywhere after |
| Raw dispersion is confounded by degree and industry | §4 peer percentiles; raw values are never the reportable quantity |
| `strength = in + out` double-counts — a weight, not a volume figure | never summed across nodes |

> **Population warning.** At P99_9 the graph is overwhelmingly individual
> nodes. Every business-facing result below filters
> `entity_type = 'business'`. §1 recomputes the exact split under
> `party_type`, which is the first thing to read.

---
## 0. Setup

In [ ]:
import os, math, warnings, textwrap
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# Plotly renderer. 'notebook' keeps the figures inside the .ipynb; if the
# figures do not appear on JupyterHub, switch to 'iframe'.
import plotly.io as pio
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"
PALETTE = px.colors.qualitative.Safe

# ---------------------------------------------------------------- config --
# Two ways in. The parquet path is what the pipeline writes; the Hive table is
# the same data registered. Parquet is usually faster (no metastore round
# trip) and works before the table is refreshed.
SOURCE       = "table"         # "table" | "parquet"
HIVE_TABLE   = "bdahd01p_dlcdi1_cdi_tm.cust_c2c_metrics"
PARQUET_GLOB = "/user/pk36814/metrics/node/*.parquet"

VERSION     = "P99_9"          # primary production rung
VERSION_ALT = "P99"            # adjacent rung, for the §11 agreement check
REF_MONTH   = None             # None -> latest month found in §1
BIZ         = "business"       # entity_type value for organisations

# Analysis gates (justified in §2; change here, not inline)
MIN_GEO_COV   = 0.60           # min located-dollar share for a geo result
MIN_CP        = 5              # min counterparties (matches the entropy guard)
PERSIST_M     = 3              # consecutive months required to call a signal
PANEL_TOP_N   = 150_000        # business nodes kept in the temporal panel

# Prototyping knob. Set to None for the full pull once the notebook runs clean.
SAMPLE_LIMIT  = 400_000

OUT = "../metrics/eda_geo"
os.makedirs(OUT, exist_ok=True)
print(f"source={SOURCE}\nversion={VERSION}  alt={VERSION_ALT}  sample_limit={SAMPLE_LIMIT}")

In [ ]:
# ------------------------------------------------- Spark data access -----
# 96.5M rows across all rungs. Everything heavy stays in Spark; only
# aggregated or explicitly-filtered results are pulled into pandas.
import time
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("pkg_geo_eda")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.execution.arrow.maxRecordsPerBatch", "50000")
         .config("spark.sql.shuffle.partitions", "400")
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

SDF = (spark.table(HIVE_TABLE) if SOURCE == "table"
       else spark.read.parquet(PARQUET_GLOB))

# One temp view for the whole notebook. Every SQL cell below reads FROM {TABLE},
# so pointing TABLE at the view means the same SQL runs against either source.
SDF.createOrReplaceTempView("pkg_metrics")
TABLE = "pkg_metrics"

n_all = SDF.count()
print(f"{SOURCE}: {n_all:,} rows x {len(SDF.columns)} columns")

def q(sql, label="", max_rows=5_000_000):
    """Run Spark SQL and return pandas.

    Guarded on purpose: a stray SELECT * against 96.5M rows would try to
    collect the whole table to the driver. Aggregate in Spark, land small
    frames in pandas — that is the whole discipline of this notebook.
    """
    t = time.time()
    sdf = spark.sql(sql)
    n = sdf.count()
    if n > max_rows:
        raise MemoryError(
            f"[{label}] {n:,} rows exceeds max_rows={max_rows:,}. Aggregate "
            f"in Spark first, or raise max_rows deliberately if the driver "
            f"can hold it.")
    df = sdf.toPandas()
    mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"[{label or 'query'}] {len(df):,} rows x {df.shape[1]} cols "
          f"| {time.time()-t:,.1f}s | {mb:,.1f} MB")
    return df

# Never downcast a coordinate. float16 carries ~3 significant digits, so
# lat 40.44052 becomes 40.4375 — a silent 0.3 km displacement on EVERY node,
# which would corrupt every distance, centroid and drift figure downstream.
NEVER_SHRINK = {"lat", "lon", "geo_centroid_lat", "geo_centroid_lon"}

def shrink(df):
    """Downcast a collected frame. toPandas returns float64/object for
    everything; on a multi-million-row panel that is most of the memory.

    float32 is the floor, never float16: numpy's linalg rejects float16
    outright (so §4 would crash), and where it does not crash it quietly
    destroys precision.
    """
    for c in df.columns:
        if c in NEVER_SHRINK:
            df[c] = df[c].astype("float64")
        elif df[c].dtype == "float64":
            df[c] = df[c].astype("float32")
        elif df[c].dtype == "int64":
            df[c] = pd.to_numeric(df[c], downcast="integer")
        elif df[c].dtype == "object" and df[c].nunique(dropna=True) < len(df) / 4:
            df[c] = df[c].astype("category")
    return df

### 0.1 Schema discovery

Column names are resolved from the table itself rather than assumed. Anything
this notebook needs but cannot find is reported here, and the dependent
sections degrade rather than raising `KeyError` twenty cells later.

In [ ]:
DTYPES  = dict(SDF.dtypes)
ALLCOLS = [c.lower() for c in SDF.columns]
print(f"{len(ALLCOLS)} columns\n")
print(pd.Series(DTYPES).value_counts().rename("n_columns").to_string())

def have(*cands, quiet=False):
    """First candidate present in the table, else None."""
    for c in cands:
        if c in ALLCOLS:
            return c
    if not quiet:
        print(f"  [missing] none of {cands}")
    return None

def haveall(prefix):
    return sorted(c for c in ALLCOLS if c.startswith(prefix))

C = {
    "time":      have("time_key", "month", "yyyymm"),
    "version":   have("version", "ladder_version"),
    "node":      have("node", "mdm_id"),
    # typing / identity
    "node_type": have("node_type"),
    "etype":     have("entity_type"),
    "etype_obs": have("entity_type_observed"),
    "etype_inf": have("entity_type_inferred"),
    "esource":   have("entity_type_source"),
    "eclass":    have("entity_class"),
    "naics2":    have("naics2"),
    "naics_desc":have("naics_desc"),
    "cust_name": have("cust_name", "customer_name"),
    "state":     have("state"),
    "zip3":      have("zip3"),
    "lat":       have("lat"), "lon": have("lon"),
    "geo_status":have("geo_status"),
    "attr_prof": have("attr_profile"),
    # flow
    "in_s":      have("in_strength"), "out_s": have("out_strength"),
    "in_d":      have("in_degree"),   "out_d": have("out_degree"),
    "deg":       have("degree"), "net": have("net_flow"),
    # concentration / structure
    "top_share": have("top_share", "top1_share", quiet=True),
    "pagerank":  have("pagerank", "pagerank_logw", quiet=True),
    "clustering":have("clustering_coef", "clustering", quiet=True),
    "recip":     have("reciprocity_node_w", "reciprocity", quiet=True),
    "hub_exp":   have("hub_exposure", quiet=True),
    "months_act":have("months_active", quiet=True),
}
GEO   = haveall("geo_")
SHARE = haveall("share_")
NAICSM= [c for c in ALLCOLS if c.startswith("naics2_") or c.startswith("same_naics2")]

print(f"\ngeo_* columns  ({len(GEO)}): {GEO}")
print(f"\nshare_* columns ({len(SHARE)})")
print(f"naics mix cols : {NAICSM}")
missing = [k for k, v in C.items() if v is None]
print(f"\nunresolved keys: {missing if missing else 'none'}")

In [ ]:
# ------------------------------------------------------------- helpers --
R_EARTH_KM = 6371.0088

def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorised great-circle distance. Used for centroid drift (§6) and
    the registered-vs-flow gap cross-check (§8)."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = p2 - p1
    dl = np.radians(np.asarray(lon2) - np.asarray(lon1))
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R_EARTH_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def sdiv(a, b, fill=np.nan):
    """Safe divide. np.where(cond, a/b, x) still evaluates a/b everywhere
    and emits the warning anyway — this does not."""
    a = np.asarray(a, dtype="float64"); b = np.asarray(b, dtype="float64")
    out = np.full(a.shape, fill, dtype="float64")
    ok = np.isfinite(a) & np.isfinite(b) & (b != 0)
    np.divide(a, b, out=out, where=ok)
    return out

def geo_gate(df, cov_cols=("geo_cov_amt_in", "geo_cov_amt_out"),
             min_cov=None, min_cp=None, biz_only=True, verbose=True):
    """The standing filter for any geographic result.

    Three independent reasons a geo value is not analysable:
      - the node is not a business (the graph is household-dominated),
      - too little of its flow reached a LOCATED counterparty,
      - too few counterparties for the dispersion guards to be meaningful.
    Applied as one function so no section quietly forgets one.
    """
    min_cov = MIN_GEO_COV if min_cov is None else min_cov
    min_cp  = MIN_CP if min_cp is None else min_cp
    m = pd.Series(True, index=df.index)
    steps = [("start", int(m.sum()))]
    if biz_only and C["etype"] in df:
        m &= df[C["etype"]].eq(BIZ); steps.append(("business", int(m.sum())))
    cov = [c for c in cov_cols if c in df]
    if cov:
        m &= df[cov].max(axis=1).ge(min_cov)
        steps.append((f"geo_cov>={min_cov}", int(m.sum())))
    ncp = [c for c in ("geo_n_cp_located_in", "geo_n_cp_located_out") if c in df]
    if ncp:
        m &= df[ncp].sum(axis=1).ge(min_cp)
        steps.append((f"n_cp>={min_cp}", int(m.sum())))
    if verbose:
        print(" -> ".join(f"{k}: {v:,}" for k, v in steps))
    return df.loc[m]

---
## 1. Data contract gates

Nothing below §1 means anything if these fail. Four checks:

1. **Panel completeness** — 23 months present at this rung, no month
   silently short.
2. **Population composition under `party_type`** — the headline
   individual-share figure was measured on the old name typer and can only
   move down. This is where it gets recomputed.
3. **Geographic coverage** — node-level and dollar-weighted.
4. **Guard-NaN rates** — how much of the geo block is actually populated,
   which determines the analysable population for everything after.

In [ ]:
sql = f"""
SELECT {C['time']} AS time_key,
       {C['version']} AS version,
       COUNT(*)                                   AS n_rows,
       COUNT(DISTINCT {C['node']})                AS n_nodes,
       SUM({C['in_s']})                           AS tot_in_strength,
       SUM({C['out_s']})                          AS tot_out_strength
FROM {TABLE}
GROUP BY 1, 2
ORDER BY 2, 1
"""
panel = q(sql, "panel completeness")
piv = panel.pivot(index="time_key", columns="version", values="n_nodes")
print(piv.to_string())
print("\nmonths per version:\n", panel.groupby("version")["time_key"].nunique().to_string())

MONTHS = sorted(panel.loc[panel.version == VERSION, "time_key"].astype(str))
REF_MONTH = REF_MONTH or MONTHS[-1]
print(f"\n{len(MONTHS)} months at {VERSION}: {MONTHS[0]} .. {MONTHS[-1]}   REF_MONTH={REF_MONTH}")

# in_strength and out_strength must agree in a closed on-us system
tot = panel[panel.version == VERSION]
resid = sdiv(tot.tot_in_strength - tot.tot_out_strength, tot.tot_in_strength)
print(f"closure residual (in vs out): max |{np.nanmax(np.abs(resid)):.2e}| "
      f"— should be ~0 for an internally closed graph")

In [ ]:
fig = px.line(panel, x="time_key", y="n_nodes", color="version", markers=True,
              title="Node count by ablation rung — panel completeness check",
              color_discrete_sequence=PALETTE)
fig.update_layout(height=380, xaxis_title=None, yaxis_title="distinct nodes",
                  legend_title="rung")
fig.show()

### 1.2 Population composition under `party_type` — recompute before citing

The previous headline (94.3% individual by node, 48.4% of dollars) was
measured when `entity_type` was inferred from `customer_name`. Under
`party_type` the correction runs **one way only** — person-named
organisations (trusts, estates, single-member LLCs) move individual →
business, never the reverse. This cell produces the number that replaces it.

In [ ]:
sql = f"""
SELECT {C['time']}      AS time_key,
       {C['node_type']} AS node_type,
       COUNT(*)                         AS n_nodes,
       SUM({C['in_s']} + {C['out_s']})  AS strength
FROM {TABLE}
WHERE {C['version']} = '{VERSION}'
GROUP BY 1, 2
"""
comp = q(sql, "composition by node_type")
comp["node_share"]  = comp.groupby("time_key")["n_nodes"].transform(lambda s: s / s.sum())
comp["dollar_share"] = comp.groupby("time_key")["strength"].transform(lambda s: s / s.sum())

latest = comp[comp.time_key.astype(str) == REF_MONTH].sort_values("n_nodes", ascending=False)
print(f"--- {REF_MONTH} @ {VERSION} ---")
print(latest[["node_type", "n_nodes", "node_share", "dollar_share"]].to_string(index=False))
ind = latest.loc[latest.node_type == "individual"]
if len(ind):
    print(f"\nINDIVIDUAL SHARE: {ind.node_share.iloc[0]:.2%} of nodes, "
          f"{ind.dollar_share.iloc[0]:.2%} of dollars")
    print("Compare against the pre-party_type figures (94.26% / 48.4%). "
          "The delta is the person-named-organisation correction.")

In [ ]:
long = comp.melt(id_vars=["time_key", "node_type"],
                 value_vars=["node_share", "dollar_share"],
                 var_name="basis", value_name="share")
fig = px.area(long.sort_values("time_key"), x="time_key", y="share",
              color="node_type", facet_col="basis",
              category_orders={"basis": ["node_share", "dollar_share"]},
              title=f"Population composition over time @ {VERSION} — "
                    f"nodes vs dollars (scope: on_us_c2c)",
              color_discrete_sequence=PALETTE)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=430, xaxis_title=None, legend_title="node_type")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [ ]:
# How much did party_type actually change? Observed vs inferred, one month.
if C["etype_obs"] and C["etype_inf"]:
    sql = f"""
    SELECT {C['etype_obs']} AS observed, {C['etype_inf']} AS inferred,
           COUNT(*) AS n, SUM({C['in_s']} + {C['out_s']}) AS strength
    FROM {TABLE}
    WHERE {C['version']} = '{VERSION}' AND {C['time']} = '{REF_MONTH}'
    GROUP BY 1, 2
    """
    dis = q(sql, "observed vs inferred")
    dis["pct_nodes"] = dis.n / dis.n.sum()
    dis["pct_dollars"] = dis.strength / dis.strength.sum()
    mat = dis.pivot(index="observed", columns="inferred", values="pct_nodes").fillna(0)
    fig = px.imshow(mat, text_auto=".2%", aspect="auto", color_continuous_scale="Blues",
                    title="Entity type: declared (party_type) vs inferred (name) "
                          f"— share of nodes, {REF_MONTH}")
    fig.update_layout(height=380, xaxis_title="inferred from customer_name",
                      yaxis_title="observed from party_type")
    fig.show()
    key = dis[(dis.observed == "business") & (dis.inferred == "individual")]
    if len(key):
        print(f"PERSON-NAMED ORGANISATIONS: {key.pct_nodes.iloc[0]:.2%} of nodes, "
              f"{key.pct_dollars.iloc[0]:.2%} of dollars.")
        print("These were counted as households under the old typer. They are "
              "trusts, estates, single-member LLCs and sole proprietorships — "
              "and they are a segment in their own right, not just an error bar.")

### 1.3 Geographic coverage and guard-NaN rates

Every Block F metric is computed on **located counterparties only**, and the
dispersion metrics are deliberately NaN below their guards (`geo_spread_*`
below 2 counterparties, entropy below 5). The analysable population is
therefore smaller than the node count, and by how much is not optional
context — it is the denominator for everything in §7–§10.

In [ ]:
gsel = [c for c in ["geo_spread_km", "geo_zip3_entropy", "geo_reach_mean_km",
                    "geo_reach_p90_km", "geo_registered_vs_flow_km",
                    "geo_home_zip3_share_in", "geo_home_state_share_in",
                    "geo_cov_amt_in", "geo_cov_amt_out"] if c in GEO]
nn = ",\n       ".join(
    f"SUM(CASE WHEN {c} IS NOT NULL THEN 1 ELSE 0 END) AS nn_{c}" for c in gsel)
sql = f"""
SELECT {C['node_type']} AS node_type,
       COUNT(*) AS n_nodes,
       {nn},
       AVG({'geo_cov_amt_in' if 'geo_cov_amt_in' in GEO else 'NULL'}) AS avg_cov_in,
       AVG({'geo_cov_amt_out' if 'geo_cov_amt_out' in GEO else 'NULL'}) AS avg_cov_out
FROM {TABLE}
WHERE {C['version']} = '{VERSION}' AND {C['time']} = '{REF_MONTH}'
GROUP BY 1
"""
cov = q(sql, "guard-NaN rates")
for c in gsel:
    cov[f"pop_{c}"] = cov[f"nn_{c}"] / cov.n_nodes
popcols = [f"pop_{c}" for c in gsel]
print(cov[["node_type", "n_nodes", "avg_cov_in", "avg_cov_out"] + popcols].to_string(index=False))

hm = cov.set_index("node_type")[popcols].rename(columns=lambda c: c.replace("pop_geo_", ""))
fig = px.imshow(hm, text_auto=".0%", aspect="auto", color_continuous_scale="Greens",
                title=f"Geo block: share of nodes with a non-null value, by node_type "
                      f"({REF_MONTH} @ {VERSION})")
fig.update_layout(height=340, xaxis_title=None, yaxis_title=None)
fig.show()
print("\nA low bar here is a GUARD, not a data-quality failure: spread is NaN "
      "below 2 counterparties and entropy below 5, deliberately.")

---
## 2. The reference cross-section

One month, all node types, the columns the rest of the notebook needs. Pulled
once and reused. Everything geographic downstream passes through `geo_gate()`.

In [ ]:
base = [C[k] for k in ("node","time","node_type","etype","etype_obs","eclass",
                       "naics2","naics_desc","cust_name","state","zip3","lat","lon",
                       "geo_status","attr_prof","in_s","out_s","in_d","out_d",
                       "net","top_share","pagerank","clustering","recip",
                       "hub_exp","months_act") if C.get(k)]
want = list(dict.fromkeys(base + GEO + SHARE + NAICSM))
sel  = ",\n       ".join(want)
lim  = f"LIMIT {SAMPLE_LIMIT}" if SAMPLE_LIMIT else ""

sql = f"""
SELECT {sel}
FROM {TABLE}
WHERE {C['version']} = '{VERSION}'
  AND {C['time']}    = '{REF_MONTH}'
{lim}
"""
X = q(sql, f"cross-section {REF_MONTH}")
X.columns = [c.lower() for c in X.columns]
X = shrink(X)
for c in (C["node_type"], C["etype"], C["naics2"], C["state"], "geo_locality_class"):
    if c and c in X: X[c] = X[c].astype("category")
X["strength"] = X[C["in_s"]].fillna(0) + X[C["out_s"]].fillna(0)
X["deg_tot"]  = X[C["in_d"]].fillna(0) + X[C["out_d"]].fillna(0)
print(X.shape)
X.head(3)

In [ ]:
# The analysable business population, and what each gate costs.
G = geo_gate(X)
print(f"\nanalysable business population: {len(G):,} of {len(X):,} sampled rows")
print(f"share of business dollars retained: "
      f"{G.strength.sum() / max(X.loc[X[C['etype']].eq(BIZ), 'strength'].sum(), 1):.1%}")

### 2.1 Is coverage confounding dispersion?

`geo_spread_km` is computed over located counterparties. If located
counterparties are systematically nearer or farther than unlocated ones,
spread and coverage will be correlated and the `MIN_GEO_COV` gate is doing
real work rather than being decorative. Worth knowing which.

In [ ]:
if "geo_cov_amt_in" in X and "geo_spread_km" in X:
    d = X[X[C["etype"]].eq(BIZ)].dropna(subset=["geo_spread_km", "geo_cov_amt_in"]).copy()
    d["cov_bin"] = pd.cut(d.geo_cov_amt_in, np.arange(0, 1.05, 0.1))
    agg = (d.groupby("cov_bin", observed=True)
             .agg(n=("geo_spread_km", "size"),
                  median_spread=("geo_spread_km", "median"),
                  p90_spread=("geo_spread_km", lambda s: s.quantile(0.90)))
             .reset_index())
    agg["cov_bin"] = agg.cov_bin.astype(str)
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_bar(x=agg.cov_bin, y=agg.n, name="nodes", marker_color="#cfd8dc")
    fig.add_scatter(x=agg.cov_bin, y=agg.median_spread, name="median spread km",
                    mode="lines+markers", line=dict(width=3), secondary_y=True)
    fig.add_scatter(x=agg.cov_bin, y=agg.p90_spread, name="p90 spread km",
                    mode="lines+markers", line=dict(dash="dot"), secondary_y=True)
    fig.add_vline(x=int(MIN_GEO_COV * 10) - 0.5, line_dash="dash", line_color="crimson")
    fig.update_layout(height=400, title="Does located-dollar coverage bias measured "
                      "dispersion? (business nodes)", xaxis_title="geo_cov_amt_in bin")
    fig.update_yaxes(title_text="nodes", secondary_y=False)
    fig.update_yaxes(title_text="km", secondary_y=True)
    fig.show()
    lo = d[d.geo_cov_amt_in < MIN_GEO_COV].geo_spread_km.median()
    hi = d[d.geo_cov_amt_in >= MIN_GEO_COV].geo_spread_km.median()
    print(f"median spread below the gate: {lo:,.0f} km | at or above: {hi:,.0f} km "
          f"| ratio {sdiv(lo, hi):.2f}")
    print("A ratio far from 1.0 means low-coverage nodes are not a random "
          "subsample and the gate is load-bearing. Near 1.0 means the gate is "
          "cheap insurance and can be relaxed if it costs too much population.")

---
## 3. What the geographic block looks like

Distributions first, by `node_type`, because the households and the businesses
are different populations and a pooled histogram is a statement about
households.

In [ ]:
dist_cols = [c for c in ["geo_spread_km", "geo_reach_p50_km", "geo_reach_p90_km",
                         "geo_zip3_entropy", "geo_n_zip3_80",
                         "geo_registered_vs_flow_km"] if c in X]
d = X[X[C["node_type"]].isin(["individual", "business_naics_valid",
                              "business_naics_missing"])]
fig = make_subplots(rows=2, cols=3, subplot_titles=dist_cols)
for i, c in enumerate(dist_cols):
    r, k = divmod(i, 3)
    for j, nt in enumerate(d[C["node_type"]].cat.remove_unused_categories().cat.categories):
        v = d.loc[d[C["node_type"]].eq(nt), c].dropna()
        if len(v) < 100: continue
        v = np.log10(v.clip(lower=0.1)) if c.endswith("_km") else v
        fig.add_histogram(x=v, name=str(nt), legendgroup=str(nt),
                          showlegend=(i == 0), opacity=0.55, nbinsx=60,
                          marker_color=PALETTE[j], row=r + 1, col=k + 1)
fig.update_layout(height=620, barmode="overlay",
                  title="Geo block distributions by node_type "
                        f"({REF_MONTH} @ {VERSION}) — km axes are log10")
fig.show()
print("Read the km panels as log10: 2.0 = 100 km, 3.0 = 1,000 km.")

In [ ]:
if "geo_locality_class" in X and C["naics2"]:
    b = X[X[C["etype"]].eq(BIZ)].dropna(subset=["geo_locality_class"])
    top = b[C["naics2"]].value_counts().head(14).index
    ct = (pd.crosstab(b.loc[b[C['naics2']].isin(top), C["naics2"]],
                      b.loc[b[C['naics2']].isin(top), "geo_locality_class"],
                      normalize="index")
          .reindex(columns=["LOCAL", "REGIONAL", "MULTI_MARKET", "NATIONAL"]))
    fig = px.imshow(ct, text_auto=".0%", aspect="auto", color_continuous_scale="Purples",
                    title="Locality class by NAICS2 sector — business nodes. "
                          "The row profile IS the industry footprint signature")
    fig.update_layout(height=520, xaxis_title=None, yaxis_title="naics2")
    fig.show()
    print("Rows that look alike are sectors with the same footprint shape. A "
          "customer whose class disagrees with its sector row is the anomaly "
          "candidate — that is the seed of the peer percentile in §4.")

---
## 4. Peer normalisation — the missing layer

The manifest is explicit that **raw dispersion is not the reportable
quantity**: a node with four counterparties *cannot* have high ZIP3 entropy,
and a large firm mechanically reaches farther. Block F emits raw values and
leaves normalisation as a downstream step. This section is that step.

**Method.** For each geo metric:

1. Regress `log1p(metric)` on `log1p(degree)` and `log1p(strength)` — OLS,
   fitted **within `node_type`** so a household's size-distance relationship
   is not imposed on a business.
2. Take the residual — the part of the footprint not explained by size.
3. Percentile-rank the residual within **`naics2` × size-decile ×
   `node_type`**, requiring a minimum peer-group size.

The output `{metric}_pctile_naics_size` is what any downstream alert,
scorecard or app should consume. Raw km values are for display only.

In [ ]:
from numpy.linalg import lstsq

PEER_MIN = 30      # smallest peer group we will percentile within

def peer_percentile(df, metric, size_cols=None, peer_cols=None,
                    peer_min=PEER_MIN, logy=True):
    """Residualise `metric` on size, then percentile-rank within peers.

    Returns (residual, percentile). Percentile is NaN where the peer group is
    too small to rank against — an unrankable node is not a median node, and
    filling it with 0.5 would invent a peer comparison that was never made.
    """
    size_cols = size_cols or ["deg_tot", "strength"]
    peer_cols = peer_cols or [C["naics2"], "size_decile", C["node_type"]]
    d = df[[metric] + size_cols + [c for c in peer_cols if c in df]].copy()
    y_ok = d[metric].notna() & np.isfinite(d[metric])
    y = (np.log1p(d.loc[y_ok, metric].clip(lower=0)) if logy
         else d.loc[y_ok, metric]).astype("float64")
    # float64 explicitly: lstsq refuses float16 and loses conditioning on
    # float32, and a downcast frame can arrive here as either.
    Xd = np.column_stack([np.ones(y_ok.sum())] +
                         [np.log1p(d.loc[y_ok, c].fillna(0).clip(lower=0)
                                   ).astype("float64") for c in size_cols])
    beta, *_ = lstsq(Xd, y.to_numpy(), rcond=None)
    resid = pd.Series(np.nan, index=d.index)
    resid.loc[y_ok] = y.to_numpy() - Xd @ beta
    r2 = 1 - np.nanvar(resid.loc[y_ok]) / max(np.nanvar(y), 1e-12)
    grp = d.assign(_r=resid).groupby([c for c in peer_cols if c in d], observed=True)["_r"]
    pct = grp.rank(pct=True)
    pct[grp.transform("size") < peer_min] = np.nan
    return resid, pct, r2, beta

B = G.copy()
B["size_decile"] = pd.qcut(B.strength.rank(method="first"), 10,
                           labels=False, duplicates="drop")
metrics = [c for c in ["geo_spread_km", "geo_reach_p50_km", "geo_zip3_entropy",
                       "geo_n_zip3_80", "geo_spread_in_km", "geo_spread_out_km"]
           if c in B]
rows = []
for m in metrics:
    r, p, r2, beta = peer_percentile(B, m)
    B[m + "_resid"] = r
    B[m + "_pctile_naics_size"] = p
    rows.append({"metric": m, "R2_size_explains": r2,
                 "beta_log_degree": beta[1], "beta_log_strength": beta[2],
                 "rankable": int(p.notna().sum()),
                 "rankable_pct": p.notna().mean()})
norm = pd.DataFrame(rows)
print(norm.to_string(index=False))
print("\nR2_size_explains is the share of the raw metric that is just size. "
      "The higher it is, the more misleading the raw value would have been.")

In [ ]:
# Before / after: how much of the size confound did we actually remove?
chk = []
for m in metrics:
    chk.append({"metric": m,
                "raw_vs_log_strength": B[m].corr(np.log1p(B.strength), method="spearman"),
                "resid_vs_log_strength": B[m + "_resid"].corr(np.log1p(B.strength),
                                                              method="spearman"),
                "raw_vs_log_degree": B[m].corr(np.log1p(B.deg_tot), method="spearman"),
                "resid_vs_log_degree": B[m + "_resid"].corr(np.log1p(B.deg_tot),
                                                            method="spearman")})
chk = pd.DataFrame(chk)
print(chk.to_string(index=False))

plot = chk.melt(id_vars="metric", var_name="pair", value_name="spearman")
fig = px.bar(plot, x="metric", y="spearman", color="pair", barmode="group",
             title="Size confounding before and after residualisation "
                   "(Spearman vs log size)", color_discrete_sequence=PALETTE)
fig.add_hline(y=0, line_color="black")
fig.update_layout(height=420, xaxis_title=None)
fig.show()

---
## 5. Does geography say anything the other metrics don't?

The honest test. If `geo_spread_pctile` is just `degree` in disguise, there is
no geographic product here. Spearman on the **peer-normalised** values,
business nodes only, coverage-gated.

In [ ]:
corr_cols = ([m + "_pctile_naics_size" for m in metrics if m + "_pctile_naics_size" in B]
             + [c for c in ["geo_home_zip3_share_in", "geo_home_state_share_in",
                            "geo_home_state_share_out", "geo_R",
                            "geo_registered_vs_flow_km", "geo_cov_amt_in"] if c in B]
             + [c for c in [C["top_share"], C["pagerank"], C["clustering"],
                            C["recip"], C["hub_exp"]] if c and c in B]
             + [c for c in B.columns if c.startswith("share_in_amt_")
                or c.startswith("share_out_amt_")][:6]
             + [c for c in ["naics2_entropy_in", "naics2_entropy_out",
                            "same_naics2_in_share", "same_naics2_out_share"] if c in B]
             + ["strength", "deg_tot"])
corr_cols = [c for c in dict.fromkeys(corr_cols) if c in B]
cm = B[corr_cols].corr(method="spearman", min_periods=500)
short = {c: c.replace("_pctile_naics_size", "·pct").replace("geo_", "")
             .replace("share_", "sh_").replace("_amt", "") for c in corr_cols}
fig = px.imshow(cm.rename(index=short, columns=short), text_auto=".2f",
                color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto",
                title="Spearman correlation — peer-normalised geo vs the rest of "
                      f"the metric set (business, coverage-gated, {REF_MONTH})")
fig.update_layout(height=780)
fig.show()

In [ ]:
# The specific question: what does footprint width relate to, once size is out?
tgt = "geo_spread_km_pctile_naics_size"
if tgt in cm:
    s = cm[tgt].drop(labels=[tgt]).dropna().sort_values()
    top = pd.concat([s.head(8), s.tail(8)]).rename("rho").reset_index()
    top.columns = ["metric", "rho"]
    top["label"] = [short.get(i, i) for i in top.metric]
    fig = px.bar(top, x="rho", y="label", orientation="h", color="rho",
                 color_continuous_scale="RdBu_r", range_color=[-0.6, 0.6],
                 title="Strongest associations with peer-normalised footprint width")
    fig.update_layout(height=520, xaxis_title="Spearman rho", yaxis_title=None,
                      coloraxis_showscale=False)
    fig.show()
    print(s.to_string())
    print("\nIf |rho| against strength and degree is now near zero while the "
          "composition and homophily terms are not, the peer-normalised "
          "footprint is carrying independent information — which is the "
          "precondition for any of the use cases below being worth building.")

---
## 6. Temporal dynamics — the panel

23 months of footprint per node. This is the part that cannot be done in a
tabular warehouse without the graph, and it is where the usable products are.

The panel is restricted to the largest `PANEL_TOP_N` business nodes: the
household tail contributes rows, not signal, and the panel is the one pull
that has to fit in memory.

In [ ]:
pcols = [c for c in [C["node"], C["time"], C["node_type"], C["etype"], C["naics2"],
                     C["state"], C["cust_name"], C["in_s"], C["out_s"],
                     C["in_d"], C["out_d"], C["lat"], C["lon"]] if c]
pgeo  = [c for c in ["geo_spread_km", "geo_spread_in_km", "geo_spread_out_km",
                     "geo_centroid_lat", "geo_centroid_lon", "geo_locality_class",
                     "geo_zip3_entropy", "geo_n_zip3_80", "geo_reach_p50_km",
                     "geo_registered_vs_flow_km", "geo_home_state_share_in",
                     "geo_home_state_share_out", "geo_cov_amt_in", "geo_cov_amt_out",
                     "geo_n_cp_located_in", "geo_n_cp_located_out"] if c in GEO]
sel = ",\n       ".join(dict.fromkeys(pcols + pgeo))

# Rank business nodes once by mean strength, keep the top N across all months.
sql = f"""
WITH ranked AS (
  SELECT {C['node']} AS node,
         AVG({C['in_s']} + {C['out_s']}) AS avg_strength
  FROM {TABLE}
  WHERE {C['version']} = '{VERSION}' AND {C['etype']} = '{BIZ}'
  GROUP BY 1
  ORDER BY 2 DESC
  LIMIT {PANEL_TOP_N}
)
SELECT m.{sel.replace(chr(10) + '       ', chr(10) + '       m.')}
FROM {TABLE} m
JOIN ranked r ON m.{C['node']} = r.node
WHERE m.{C['version']} = '{VERSION}'
"""
P = q(sql, "temporal panel", max_rows=8_000_000)
P.columns = [c.lower() for c in P.columns]
P = shrink(P)
P = P.rename(columns={C["node"]: "node", C["time"]: "time_key"})
P["time_key"] = P.time_key.astype(str)
P["strength"] = P[C["in_s"]].fillna(0) + P[C["out_s"]].fillna(0)
P["deg_tot"]  = P[C["in_d"]].fillna(0) + P[C["out_d"]].fillna(0)
P = P.sort_values(["node", "time_key"])
print(f"panel: {P.node.nunique():,} business nodes x {P.time_key.nunique()} months")

### 6.1 Locality-class transitions

A Markov transition matrix on `geo_locality_class`, month over month. The
diagonal is stickiness. **If the diagonal is not dominant, the class is
noise** and every alert built on it will be noise too — so this is a
go/no-go for §7, not a description.

In [ ]:
if "geo_locality_class" in P:
    t = P[["node", "time_key", "geo_locality_class"]].dropna()
    t["nxt"] = t.groupby("node")["geo_locality_class"].shift(-1)
    t = t.dropna(subset=["nxt"])
    order = ["LOCAL", "REGIONAL", "MULTI_MARKET", "NATIONAL"]
    M = (pd.crosstab(t.geo_locality_class, t.nxt, normalize="index")
           .reindex(index=order, columns=order))
    fig = px.imshow(M, text_auto=".1%", color_continuous_scale="Blues",
                    title="Locality-class transition matrix, month over month "
                          "(business nodes) — diagonal = stickiness")
    fig.update_layout(height=430, xaxis_title="next month", yaxis_title="this month")
    fig.show()
    diag = float(np.nanmean(np.diag(M.to_numpy())))
    print(f"mean diagonal = {diag:.1%}")
    print("Above ~85%: the class is stable enough that a transition is an event "
          "worth alerting on. Below ~70%: the class is churning and §7 must use "
          "the continuous spread series with a persistence rule instead.")

### 6.2 Footprint trajectories — expansion, contraction, drift

Three distinct dynamics, deliberately measured separately:

- **Spread trend** — is the counterparty cloud widening? Theil–Sen slope on
  `log1p(geo_spread_km)`, which is robust to the one-month spikes a single
  large remote payment produces.
- **Centroid drift** — is the centre of mass *moving*, independent of width?
  A firm can relocate its economic centre without widening at all.
- **Persistence** — neither counts unless it holds for `PERSIST_M`
  consecutive months. This is what separates a signal from a big invoice.

In [ ]:
def theilsen_slope(y):
    """Median pairwise slope. Robust to the single-month spike that one
    large remote payment produces in an amount-weighted metric."""
    y = np.asarray(y, dtype="float64")
    ok = np.isfinite(y)
    if ok.sum() < 4: return np.nan
    x = np.arange(len(y))[ok]; y = y[ok]
    i, j = np.triu_indices(len(y), k=1)
    dx = x[j] - x[i]
    return float(np.median((y[j] - y[i])[dx > 0] / dx[dx > 0]))

g = P.groupby("node", sort=False)
traj = pd.DataFrame({
    "n_months":     g["time_key"].size(),
    "avg_strength": g["strength"].mean(),
    "naics2":       g[C["naics2"]].first() if C["naics2"] else np.nan,
    "state":        g[C["state"]].first() if C["state"] else np.nan,
    "name":         g[C["cust_name"]].first() if C["cust_name"] else "",
})
if "geo_spread_km" in P:
    P["_lsp"] = np.log1p(P.geo_spread_km)
    traj["spread_slope"] = P.groupby("node")["_lsp"].apply(theilsen_slope)
    traj["spread_first"] = g["geo_spread_km"].first()
    traj["spread_last"]  = g["geo_spread_km"].last()
    traj["spread_med"]   = g["geo_spread_km"].median()

# centroid drift: distance between consecutive monthly centroids
if {"geo_centroid_lat", "geo_centroid_lon"} <= set(P.columns):
    P["_plat"] = P.groupby("node")["geo_centroid_lat"].shift()
    P["_plon"] = P.groupby("node")["geo_centroid_lon"].shift()
    P["drift_km"] = haversine_km(P._plat, P._plon,
                                 P.geo_centroid_lat, P.geo_centroid_lon)
    traj["drift_med_km"]   = P.groupby("node")["drift_km"].median()
    traj["drift_total_km"] = haversine_km(
        g["geo_centroid_lat"].first(), g["geo_centroid_lon"].first(),
        g["geo_centroid_lat"].last(),  g["geo_centroid_lon"].last())
    # net displacement over cumulative wandering: 1.0 = a directed move,
    # ~0 = jitter around a fixed point. This is what separates relocation
    # from month-to-month noise.
    traj["drift_directedness"] = sdiv(traj.drift_total_km,
                                      P.groupby("node")["drift_km"].sum())
traj = traj[traj.n_months >= max(PERSIST_M + 1, 6)]
print(traj.describe().T.to_string())

In [ ]:
if "spread_slope" in traj and "drift_med_km" in traj:
    s = traj.dropna(subset=["spread_slope", "drift_med_km"]).sample(
        min(20000, len(traj)), random_state=0)
    fig = px.scatter(s, x="spread_slope", y=s.drift_med_km.clip(upper=2000),
                     color=np.log10(s.avg_strength.clip(lower=1)),
                     opacity=0.35, color_continuous_scale="Viridis",
                     labels={"x": "Theil-Sen slope of log1p(spread) per month",
                             "y": "median monthly centroid drift (km)",
                             "color": "log10 avg strength"},
                     title="Two independent footprint dynamics: widening vs moving")
    fig.add_vline(x=0, line_dash="dash", line_color="grey")
    fig.update_layout(height=520)
    fig.show()
    print("The two axes are close to independent by construction — a firm can "
          "widen without moving (new remote customers around the same base) or "
          "move without widening (relocation). Alerts should treat them "
          "separately; a combined 'geo change score' would blur two different "
          "business events into one number nobody can action.")

---
## 7. USE CASE A — Footprint Expansion / Contraction Monitor
### *"Which clients' geographic footprint changed materially this quarter?"*

**Who it serves.** TM Sales (expansion = a customer entering new markets needs
multi-state treasury services, lockbox, consolidated reporting) and Risk
(persistent contraction alongside falling strength is a distress pattern).

**Why the graph.** No tabular source has a customer's counterparty geography.
The address table knows where the customer *is*; only the payment graph knows
where its *money* is.

**Definition.** A node is flagged when all four hold:

1. `PERSIST_M` consecutive months of same-signed month-over-month change in
   `log1p(geo_spread_km)`,
2. total change exceeding the peer-relative threshold (`naics2` × size
   decile), not an absolute km figure,
3. coverage gate held in **every** month of the window — a coverage change
   would otherwise masquerade as a footprint change,
4. counterparty count did not collapse (a footprint that "shrinks" because
   the customer lost counterparties is a churn signal, not a geographic one,
   and belongs in a different queue).

Condition 3 is the one that would be forgotten in a hand-rolled version and
is the most likely source of false positives.

In [ ]:
def run_lengths(sign_series):
    """Longest run of consecutive same-signed month-over-month changes."""
    out = {}
    for node, s in sign_series.groupby(level=0):
        best = cur = prev = 0
        for x in s.to_numpy():
            cur = cur + 1 if (x != 0 and x == prev) else (1 if x != 0 else 0)
            prev = x; best = max(best, cur)
        return_val = best
        out[node] = best
    return pd.Series(out)

def level_shift(df, col, k=None):
    """Persistence as a LEVEL SHIFT, not a run of same-signed diffs.

    +1 when the last k months are ALL above the trailing median of the
    earlier window, -1 when all below, 0 otherwise.

    Why not just the run rule: on a noisy series, requiring k consecutive
    same-signed first differences is close to orthogonal to a genuine trend —
    random nodes pass it about as often as drifting ones, so it costs recall
    without buying precision. A level shift against the node's own history is
    the standard test for "this changed and stayed changed", and it is what
    the business question actually asks.
    """
    k = k or PERSIST_M
    out = {}
    for node, g in df.groupby("node", sort=False):
        v = g[col].to_numpy(dtype="float64")
        v = v[np.isfinite(v)]
        if len(v) < k + 4:
            out[node] = 0; continue
        base, recent = np.median(v[:-k]), v[-k:]
        out[node] = 1 if (recent > base).all() else (-1 if (recent < base).all() else 0)
    return pd.Series(out)

W = P.copy()
W["cov_ok"] = W[[c for c in ("geo_cov_amt_in", "geo_cov_amt_out")
                 if c in W]].max(axis=1).ge(MIN_GEO_COV)
W["ncp"] = W[[c for c in ("geo_n_cp_located_in", "geo_n_cp_located_out")
              if c in W]].sum(axis=1)
W["d_lsp"] = W.groupby("node")["_lsp"].diff()
W["sgn"] = np.sign(W.d_lsp.fillna(0)).astype(int)

cand = traj.join(run_lengths(W.set_index("node")["sgn"]).rename("max_run"))
cand["level_shift"]  = level_shift(W, "_lsp")
cand["cov_ok_all"]   = W.groupby("node")["cov_ok"].all()
cand["ncp_first"]    = W.groupby("node")["ncp"].first()
cand["ncp_last"]     = W.groupby("node")["ncp"].last()
cand["ncp_retained"] = sdiv(cand.ncp_last, cand.ncp_first)

# peer-relative threshold: extreme slope WITHIN naics2 x size decile
cand["size_decile"] = pd.qcut(cand.avg_strength.rank(method="first"), 10,
                              labels=False, duplicates="drop")
grp = cand.groupby(["naics2", "size_decile"], observed=True)["spread_slope"]
cand["slope_pctile_peer"] = grp.rank(pct=True)
cand.loc[grp.transform("size") < 30, "slope_pctile_peer"] = np.nan

ok = cand.cov_ok_all & (cand.ncp_retained >= 0.7)
EXPAND   = cand[ok & (cand.level_shift == 1) & (cand.slope_pctile_peer >= 0.90)]
CONTRACT = cand[ok & (cand.level_shift == -1) & (cand.slope_pctile_peer <= 0.10)]
print(f"EXPANDING  : {len(EXPAND):,} nodes ({len(EXPAND)/len(cand):.2%} of panel)")
print(f"CONTRACTING: {len(CONTRACT):,} nodes ({len(CONTRACT)/len(cand):.2%})")
print(f"rejected by the coverage-in-every-month rule: "
      f"{int((~cand.cov_ok_all).sum()):,} ({(~cand.cov_ok_all).mean():.1%}) "
      f"— the false positives a naive version would have shipped")

# How much does each rule contribute? Overlap of the two persistence tests.
print("\npersistence rule overlap (expansion side):")
print(pd.crosstab(cand.level_shift == 1, cand.max_run >= PERSIST_M,
                  rownames=["level_shift"], colnames=[f"run>={PERSIST_M}"]).to_string())
print("\nIf the two rules barely overlap, they are testing different things. "
      "The level shift asks 'did it change and stay changed', which is the "
      "business question; the run rule asks 'did it move the same way k times "
      "in a row', which noise passes routinely. Keep level_shift as the gate "
      "and carry max_run as a descriptor only.")
EXPAND.sort_values("spread_slope", ascending=False).head(15)[
    ["name", "naics2", "state", "avg_strength", "spread_first", "spread_last",
     "spread_slope", "level_shift", "max_run", "drift_total_km"]]

In [ ]:
# Trajectories of the top expanders, against their sector median.
if len(EXPAND):
    pick = EXPAND.sort_values("avg_strength", ascending=False).head(8).index
    tr = P[P.node.isin(pick)][["node", "time_key", "geo_spread_km", C["cust_name"]]]
    med = (P.groupby("time_key")["geo_spread_km"].median()
             .rename("panel_median").reset_index())
    fig = px.line(tr, x="time_key", y="geo_spread_km", color="node", markers=True,
                  hover_data=[C["cust_name"]], color_discrete_sequence=PALETTE,
                  title="Footprint expansion candidates vs the panel median "
                        "(business nodes)")
    fig.add_scatter(x=med.time_key, y=med.panel_median, name="panel median",
                    line=dict(color="black", dash="dash", width=3))
    fig.update_layout(height=460, yaxis_title="geo_spread_km", xaxis_title=None,
                      showlegend=True)
    fig.show()

In [ ]:
# Where is the expansion happening? Net centroid displacement by state.
if "drift_total_km" in traj and C["state"]:
    mv = cand.dropna(subset=["drift_total_km"])
    st = (mv.assign(expanding=mv.index.isin(EXPAND.index))
            .groupby("state", observed=True)
            .agg(n=("drift_total_km", "size"),
                 expand_rate=("expanding", "mean"),
                 median_drift=("drift_total_km", "median")).reset_index())
    st = st[st.n >= 50]
    fig = px.choropleth(st, locations="state", locationmode="USA-states",
                        color="expand_rate", scope="usa",
                        color_continuous_scale="Tealrose",
                        hover_data=["n", "median_drift"],
                        title="Share of business nodes flagged EXPANDING, by "
                              "registered state (min 50 nodes)")
    fig.update_layout(height=520)
    fig.show()
    print(st.sort_values("expand_rate", ascending=False).head(10).to_string(index=False))

### 7.1 Calibrating the detector — placebo test

There are no labels for "this customer really expanded", so precision cannot
be measured directly. It *can* be bounded. Permuting each node's series in
time destroys any trend while preserving that node's distribution exactly, so
every flag raised on permuted data is a false positive by construction.

The flag rate on permuted data over the flag rate on real data is an empirical
**false-discovery estimate**. Tune `PERSIST_M` and the peer-percentile cutoff
until it is acceptable — do not adopt the defaults in this notebook without
running this, because they were set against synthetic data whose noise
structure is not yours.

In [ ]:
def flag_rate(panel, slope_pct_hi=0.90, persist=None):
    persist = persist or PERSIST_M
    ls = level_shift(panel, "_lsp", k=persist)
    sl = panel.groupby("node")["_lsp"].apply(theilsen_slope)
    d = pd.DataFrame({"level_shift": ls, "slope": sl}).join(
        traj[["naics2", "avg_strength"]], how="inner").dropna(subset=["slope"])
    d["size_decile"] = pd.qcut(d.avg_strength.rank(method="first"), 10,
                               labels=False, duplicates="drop")
    g = d.groupby(["naics2", "size_decile"], observed=True)["slope"]
    d["pct"] = g.rank(pct=True)
    d.loc[g.transform("size") < 30, "pct"] = np.nan
    return float(((d.level_shift == 1) & (d.pct >= slope_pct_hi)).mean())

real = flag_rate(W)
placebo = []
for seed in range(3):
    Wp = W.copy()
    r = np.random.default_rng(seed)
    Wp["_lsp"] = (Wp.groupby("node")["_lsp"]
                    .transform(lambda v: r.permutation(v.to_numpy())))
    placebo.append(flag_rate(Wp))
placebo = float(np.mean(placebo))
print(f"flag rate on REAL panel     : {real:.3%}")
print(f"flag rate on PERMUTED panel : {placebo:.3%}  (all false by construction)")
print(f"estimated false-discovery   : {sdiv(placebo, real):.1%} of flags")
print("\nAbove ~30%, tighten: raise PERSIST_M, or the peer percentile cutoff, "
      "or both, and re-run. Below ~10%, the detector is conservative and you "
      "can afford to loosen it to recover recall.")

grid = []
for pm in (2, 3, 4, 5):
    for cut in (0.90, 0.95, 0.99):
        r_ = flag_rate(W, cut, pm)
        grid.append({"persist_m": pm, "slope_cut": cut, "flag_rate": r_})
grid = pd.DataFrame(grid)
fig = px.line(grid, x="persist_m", y="flag_rate", color="slope_cut", markers=True,
              title="Detector sensitivity: flag rate vs persistence and peer cutoff",
              color_discrete_sequence=PALETTE)
fig.update_layout(height=380, yaxis_tickformat=".2%")
fig.show()

**App shape.** A monthly-refreshed queue, not a dashboard. Columns: customer,
sector, current locality class, slope percentile vs peers, months in run,
sparkline, and the top 3 new ZIP3s entering the 80% coverage set. Route
expanders to TM Sales, contractors to portfolio review. One row per customer
per event, closed when the run breaks — because a list that re-alerts the same
customer every month gets ignored by week three.

---
## 8. USE CASE B — Registered-vs-Flow Audit
### *"Is this customer's registered address where its business actually is?"*

`geo_registered_vs_flow_km` is described in the manifest as the
representativeness test, and with `addr_type` constant in MDM it is the *only*
available check on whether a registered address means anything. It is also
one column — the cheapest use case here by a wide margin.

**The interpretation is two-dimensional and this is the important part.** The
gap alone does not tell you anything; it has to be read against the spread:

| gap | spread | reading | action |
|---|---|---|---|
| small | small | address is meaningful | safe for point-level analytics |
| small | large | national player, correctly pinned at its centre | fine |
| **large** | **small** | **genuinely mislocated** — tight cluster of activity, somewhere else | **territory / servicing review** |
| large | large | HQ artifact — registered at a head office, operating everywhere | use as a label, never as a location |

Only the third row is actionable, and pooling the gap into a single ranked
list would bury it under the fourth.

In [ ]:
gapc = "geo_registered_vs_flow_km"
if gapc in G:
    A = G.dropna(subset=[gapc, "geo_spread_km"]).copy()
    A["gap_hi"]    = A[gapc] > A[gapc].quantile(0.90)
    A["spread_lo"] = A.geo_spread_km < A.geo_spread_km.median()
    A["quadrant"] = np.select(
        [~A.gap_hi & A.spread_lo, ~A.gap_hi & ~A.spread_lo,
         A.gap_hi & A.spread_lo],
        ["pinned_local", "pinned_national", "MISLOCATED"], "hq_artifact")
    qc = A.quadrant.value_counts(normalize=True)
    print(qc.to_string(), "\n")
    s = A.sample(min(25000, len(A)), random_state=0)
    fig = px.scatter(s, x=s.geo_spread_km.clip(1, 5000),
                     y=s[gapc].clip(1, 5000), color="quadrant",
                     opacity=0.4, log_x=True, log_y=True,
                     color_discrete_sequence=PALETTE,
                     labels={"x": "geo_spread_km (footprint width)",
                             "y": "registered vs flow gap (km)"},
                     title="Registered address representativeness — only the "
                           "MISLOCATED quadrant is actionable")
    fig.update_layout(height=560)
    fig.show()
    MIS = A[A.quadrant == "MISLOCATED"]
    print(f"MISLOCATED: {len(MIS):,} business nodes "
          f"({len(MIS)/len(A):.1%}), carrying "
          f"${MIS.strength.sum()/1e9:,.2f}B of strength "
          f"(strength double-counts; use as a weight, not a volume figure)")

In [ ]:
# Where would these customers be reassigned to? Registered point -> flow centroid.
if "MIS" in dir() and {"geo_centroid_lat", "geo_centroid_lon"} <= set(G.columns):
    top = MIS.nlargest(300, "strength")
    fig = go.Figure()
    for _, r in top.iterrows():
        fig.add_trace(go.Scattergeo(
            lon=[r[C["lon"]], r.geo_centroid_lon],
            lat=[r[C["lat"]], r.geo_centroid_lat],
            mode="lines", line=dict(width=1, color="rgba(200,60,60,.35)"),
            showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scattergeo(
        lon=top[C["lon"]], lat=top[C["lat"]], mode="markers",
        marker=dict(size=5, color="#1f77b4"), name="registered address",
        text=top[C["cust_name"]] if C["cust_name"] else None))
    fig.add_trace(go.Scattergeo(
        lon=top.geo_centroid_lon, lat=top.geo_centroid_lat, mode="markers",
        marker=dict(size=5, color="#d62728", symbol="diamond"),
        name="flow centroid", text=top[C["cust_name"]] if C["cust_name"] else None))
    fig.update_layout(height=620, geo=dict(scope="usa", landcolor="#f5f5f5"),
                      title="Top 300 mislocated business customers by strength — "
                            "registered pin vs where the money actually is")
    fig.show()

In [ ]:
# Territory-level implication: which states gain and lose economic presence?
if "MIS" in dir():
    nearest_state = MIS[[C["state"]]].copy()
    flows = (MIS.groupby(C["state"], observed=True)
                .agg(n=("strength", "size"), strength=("strength", "sum"))
                .sort_values("strength", ascending=False).reset_index())
    fig = px.bar(flows.head(20), x=C["state"], y="strength", hover_data=["n"],
                 title="Registered state of mislocated customers — where the "
                       "book says they are, but the flow does not",
                 color_discrete_sequence=PALETTE)
    fig.update_layout(height=400, xaxis_title=None)
    fig.show()
    print("Next step for a production version: reverse-geocode the flow centroid "
          "to a CBSA and produce the registered-CBSA -> flow-CBSA reassignment "
          "matrix. That matrix IS the territory audit deliverable; this chart is "
          "only the origin side of it.")

---
## 9. USE CASE C — Trade-Role Taxonomy from footprint asymmetry
### *"Does this customer collect locally and pay nationally, or the reverse?"*

`geo_spread_in_km` and `geo_spread_out_km` are the **revenue footprint** and
the **supply footprint**. Their ratio is a behavioural role that is
independent of NAICS, and therefore usable both as a segmentation and as a
cross-check on the industry label.

| in-spread | out-spread | role | product implication |
|---|---|---|---|
| local | local | community business | local cash management |
| local | **national** | local revenue, distant suppliers — importer / distributor / franchisee | supplier payments, FX, card |
| **national** | local | national revenue, local cost base — manufacturer / service exporter | receivables, lockbox, consolidated collection |
| national | national | national intermediary | full TM stack |

Cross-tabbing role against NAICS2 finds the customers whose behaviour
disagrees with their industry code — which is either a mislabelled NAICS or a
genuinely unusual business model, and both are worth a call.

In [ ]:
if {"geo_spread_in_km", "geo_spread_out_km"} <= set(G.columns):
    T = G.dropna(subset=["geo_spread_in_km", "geo_spread_out_km"]).copy()
    T["asym"] = np.log1p(T.geo_spread_out_km) - np.log1p(T.geo_spread_in_km)
    cut_in  = T.geo_spread_in_km.median()
    cut_out = T.geo_spread_out_km.median()
    T["role"] = np.select(
        [(T.geo_spread_in_km <= cut_in) & (T.geo_spread_out_km <= cut_out),
         (T.geo_spread_in_km <= cut_in) & (T.geo_spread_out_km > cut_out),
         (T.geo_spread_in_km > cut_in) & (T.geo_spread_out_km <= cut_out)],
        ["community", "local_revenue_distant_supply", "national_revenue_local_cost"],
        "national_intermediary")
    print(T.role.value_counts(normalize=True).to_string())
    s = T.sample(min(25000, len(T)), random_state=0)
    fig = px.scatter(s, x=s.geo_spread_in_km.clip(1, 5000),
                     y=s.geo_spread_out_km.clip(1, 5000), color="role",
                     opacity=0.35, log_x=True, log_y=True,
                     color_discrete_sequence=PALETTE,
                     labels={"x": "revenue footprint — geo_spread_in_km",
                             "y": "supply footprint — geo_spread_out_km"},
                     title="Trade-role taxonomy from footprint asymmetry "
                           "(business, coverage-gated)")
    fig.add_shape(type="line", x0=1, y0=1, x1=5000, y1=5000,
                  line=dict(dash="dash", color="grey"))
    fig.update_layout(height=580)
    fig.show()

In [ ]:
if "T" in dir() and "role" in T:
    top = T[C["naics2"]].value_counts().head(15).index
    ct = pd.crosstab(T.loc[T[C["naics2"]].isin(top), C["naics2"]],
                     T.loc[T[C["naics2"]].isin(top), "role"], normalize="index")
    fig = px.imshow(ct, text_auto=".0%", aspect="auto",
                    color_continuous_scale="Oranges",
                    title="Trade role by NAICS2 — cells far from their row's "
                          "sector norm are the mislabelled-or-interesting cases")
    fig.update_layout(height=520, xaxis_title=None, yaxis_title="naics2")
    fig.show()
    # role-vs-sector outliers: nodes whose role is rare within their own sector
    rate = ct.stack().rename("sector_role_rate").reset_index()
    rate.columns = [C["naics2"], "role", "sector_role_rate"]
    T2 = T.merge(rate, on=[C["naics2"], "role"], how="left")
    odd = T2[T2.sector_role_rate < 0.05].nlargest(20, "strength")
    print("\nCustomers whose trade role occurs in <5% of their own sector:")
    print(odd[[C["cust_name"], C["naics2"], C["naics_desc"], "role",
               "geo_spread_in_km", "geo_spread_out_km", "strength"]]
          .head(20).to_string(index=False))

---
## 10. USE CASE D — Distance-weighted counterparty concentration

Concentration and distance are separately monitored today. Together they are a
sharper risk statement: **a customer whose revenue is concentrated in a few
counterparties who are all far away** has neither diversification nor
proximity, and the relationship is harder to defend and slower to remediate.

This is a composite, so it is deliberately kept as a 2-D view rather than a
single score — a blended index would hide which of the two drove the flag.

In [ ]:
conc = C["top_share"]
if conc and conc in G and "geo_reach_p50_km" in G:
    Rk = G.dropna(subset=[conc, "geo_reach_p50_km"]).copy()
    Rk["conc_pct"]  = Rk[conc].rank(pct=True)
    Rk["reach_pct"] = Rk.geo_reach_p50_km.rank(pct=True)
    Rk["flag"] = (Rk.conc_pct >= 0.90) & (Rk.reach_pct >= 0.90)
    s = Rk.sample(min(25000, len(Rk)), random_state=0)
    fig = px.density_heatmap(s, x="reach_pct", y="conc_pct", nbinsx=40, nbinsy=40,
                             color_continuous_scale="Magma",
                             labels={"reach_pct": "distance percentile (p50 reach)",
                                     "conc_pct": "counterparty concentration percentile"},
                             title="Concentration x distance — the top-right cell is "
                                   "the queue")
    fig.add_hline(y=0.90, line_dash="dash", line_color="white")
    fig.add_vline(x=0.90, line_dash="dash", line_color="white")
    fig.update_layout(height=520)
    fig.show()
    print(f"flagged: {int(Rk.flag.sum()):,} nodes ({Rk.flag.mean():.2%}) | "
          f"strength ${Rk.loc[Rk.flag,'strength'].sum()/1e9:,.2f}B")
    print(Rk[Rk.flag].nlargest(12, "strength")[
        [C["cust_name"], C["naics2"], C["state"], conc, "geo_reach_p50_km",
         "geo_spread_km", "strength"]].to_string(index=False))
else:
    print("concentration column not resolved — skipping. Set C['top_share'] "
          "to the node-level concentration column if it exists under another name.")

---
## 11. Rung agreement — P99_9 vs P99

**Standing rule: no result is reportable until computed at two adjacent rungs
and shown to agree.** Hub removal changes who a node's counterparties are, and
the V0 → P99_9 transition has already reversed six geographic findings once.
This section repeats the two headline quantities at `P99` and compares.

In [ ]:
sql = f"""
SELECT {C['version']} AS version,
       {C['node_type']} AS node_type,
       COUNT(*) AS n_nodes,
       AVG(geo_spread_km) AS mean_spread,
       PERCENTILE_APPROX(geo_spread_km, 0.5) AS med_spread,
       PERCENTILE_APPROX(geo_reach_p50_km, 0.5) AS med_reach_p50,
       AVG(geo_home_state_share_in) AS mean_home_state_in
FROM {TABLE}
WHERE {C['version']} IN ('{VERSION}', '{VERSION_ALT}')
  AND {C['time']} = '{REF_MONTH}'
GROUP BY 1, 2
"""
rung = q(sql, "rung agreement")
piv = rung.pivot(index="node_type", columns="version")
print(piv.to_string())

cmp = rung.pivot(index="node_type", columns="version", values="med_spread")
if {VERSION, VERSION_ALT} <= set(cmp.columns):
    cmp["ratio_P99_over_P99_9"] = sdiv(cmp[VERSION_ALT], cmp[VERSION])
    print("\nmedian geo_spread_km, rung ratio by node_type:")
    print(cmp.to_string())
    print("\nRatios near 1.0 = the finding survives de-hubbing. A ratio far from "
          "1.0 for a node_type means that type's footprint was being carried by "
          "hub counterparties, and any use case above must be re-derived for it "
          "before it is briefed.")

In [ ]:
# Do the §7 candidate sets survive the rung change? Rank-correlation of the
# quantity the alert is built on.
sql = f"""
SELECT {C['node']} AS node, {C['version']} AS version,
       geo_spread_km, geo_reach_p50_km, geo_registered_vs_flow_km
FROM {TABLE}
WHERE {C['version']} IN ('{VERSION}', '{VERSION_ALT}')
  AND {C['time']} = '{REF_MONTH}'
  AND {C['etype']} = '{BIZ}'
  AND geo_spread_km IS NOT NULL
{f'LIMIT {SAMPLE_LIMIT}' if SAMPLE_LIMIT else ''}
"""
rr = q(sql, "rung rank stability")
w = rr.pivot_table(index="node", columns="version",
                   values=["geo_spread_km", "geo_reach_p50_km",
                           "geo_registered_vs_flow_km"])
out = []
for m in ["geo_spread_km", "geo_reach_p50_km", "geo_registered_vs_flow_km"]:
    if (m, VERSION) in w and (m, VERSION_ALT) in w:
        sub = w[m].dropna()
        out.append({"metric": m, "n": len(sub),
                    "spearman_P99_9_vs_P99": sub[VERSION].corr(sub[VERSION_ALT],
                                                               method="spearman")})
print(pd.DataFrame(out).to_string(index=False))
print("\nAbove ~0.95 the ranking is rung-invariant and the alert queues will be "
      "substantially the same set of customers. Below ~0.85, the metric is "
      "measuring hub structure and the use case needs a rung declared as part "
      "of its definition.")

---
## 12. What is worth building

Ranked by value per unit of effort, with what blocks each.

| # | Product | Serves | Effort | Depends on | Verdict |
|---|---|---|---|---|---|
| **B** | **Registered-vs-Flow audit** | TM Sales ops, Servicing | **Low** — one column, one quadrant rule | nothing | **Build first.** Cheapest real output here. Territory misassignment is a concrete, checkable error with an owner, and the four-quadrant reading stops it degrading into a list of large national firms |
| **A** | **Footprint Expansion / Contraction Monitor** | TM Sales, Risk | Medium — needs the panel and the persistence rule | peer normalisation (§4), locality stickiness (§6.1) | **Build second.** The genuine graph-native product. Gate it on the §6.1 diagonal: if locality class is not sticky, use the continuous series and drop the class transitions from the UI |
| **C** | **Trade-role taxonomy** | Product, TM Sales | Medium | in/out spread coverage separately gated | Build as a **segmentation attribute**, not a screen. Its value is joining to product holdings to find who has the wrong stack, which needs CRM data this notebook does not have |
| **D** | **Distance-weighted concentration** | Risk | Low | a node-level concentration column | Fold into an existing risk view as two columns. Not worth its own surface |

### Sequencing recommendation

1. **Close §4 into the pipeline.** Peer percentiles are currently computed in
   this notebook. They are the reportable quantity for three of the four use
   cases, so they belong in `pkg_geo_metrics.py` as
   `{metric}_pctile_naics_size`, written per month with the peer-group size
   recorded alongside. Until then every app re-derives them and they will
   drift apart.
2. **Ship B as a table first, not an app.** A monthly parquet of the
   MISLOCATED quadrant, reviewed by one person for one month, will tell you
   whether the flag is right far faster than a Streamlit page will.
3. **A needs a CBSA join.** Reverse-geocoding the flow centroid to a CBSA is
   what turns "moved 340 km" into "entered the Columbus market", which is the
   form Sales can act on. That is the FI Pinning Registry work — build the
   geography spine once and both B and A consume it.
4. **Then the Streamlit panel**: customer search → footprint map (registered
   pin, flow centroid, counterparty cloud), 23-month sparklines, peer
   percentile bars, and the alert history for that customer.

### What this notebook cannot tell you

- **Off-us flow.** `scope = on_us_c2c`. A customer that looks LOCAL here may
  be national through counterparties we cannot see. Every figure above is a
  statement about position *within the PNC customer base*. The expansion
  monitor in particular will systematically miss expansion into markets where
  PNC has no deposit presence — which is precisely where expansion is most
  likely. Revisit when PAYS_CPTY lands.
- **Whether the address is current.** Block C (relocation history, 660 daily
  snapshots) is built but not in the pipeline. It is the natural validator for
  use case B: a large registered-vs-flow gap that closes right after a
  recorded relocation is a stale address, not a mislocation.
- **Causality on drift.** A moving centroid can be the customer relocating,
  the customer's *counterparties* relocating, or a single large new
  relationship. Distinguishing those needs edge-level attribution, which is a
  different pull.

### Governance before any of this reaches a customer-facing surface

Geographic inference on customers touches fair-lending and CRA. A footprint
score is not a credit decision, but a queue that routes Sales attention by
geography will be asked whether it has disparate impact. Get the compliance
read **before** the Streamlit page exists, not after — and keep
`entity_type = 'business'` on every one of these surfaces, which is both the
analytically correct filter and the one that keeps consumer-protection
exposure out of scope.